# Week 03 Assignment: Gradio AI Chatbot

This project is an educational AI chatbot built with Python, Gradio, and OpenRouter.

## Features

1. Select an available language model.
2. Chat with the selected model.
3. Select multiple models for cross-analysis.
4. Compare model responses.
5. Generate a summary of similarities, differences, strengths, and weaknesses.

The application reads the OpenRouter API key from an environment variable.
No API key is stored directly in this notebook.

In [1]:
def list_free_models():

    try:
        response = requests.get(
            MODELS_URL,
            timeout=30
        )

        response.raise_for_status()

        all_models = response.json()["data"]

        free_ids = [
            model["id"]
            for model in all_models
            if model["id"].endswith(":free")
        ]

        free_ids.sort()

    except Exception as error:
        print("Could not fetch free models:", error)
        free_ids = []

    # Always keep OpenRouter's free router as a fallback
    return ["openrouter/free"] + free_ids

In [2]:
%pip install -U gradio requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import requests
import gradio as gr

from dotenv import load_dotenv


# Load variables from the local .env file.
# Do not write your real API key inside this notebook.
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODELS_URL = "https://openrouter.ai/api/v1/models"


print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))

if not OPENROUTER_API_KEY:
    print("Please check that OPENROUTER_API_KEY exists in your local .env file.")

c:\Users\ELEAZAR GIDEON\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenRouter key loaded: True


## How model selection works

The application requests the available models from OpenRouter.

Only models whose IDs end with `:free` are selected for this educational project.

The model list is fetched dynamically, so the application does not depend on one permanently fixed model.

In [4]:
def list_free_models():
    """
    Fetch available models from OpenRouter
    and return models available on the free tier.
    """

    try:
        response = requests.get(
            MODELS_URL,
            timeout=30
        )

        response.raise_for_status()

        all_models = response.json().get("data", [])

        free_models = [
            model["id"]
            for model in all_models
            if model.get("id", "").endswith(":free")
        ]

        free_models = sorted(set(free_models))

        # Include OpenRouter's automatic free-model router.
        return ["openrouter/free"] + free_models

    except requests.RequestException as error:
        print("Could not fetch free models:", error)
        return ["openrouter/free"]

    except Exception as error:
        print("Unexpected error while fetching models:", error)
        return ["openrouter/free"]


free_models = list_free_models()

print(f"Number of available choices: {len(free_models)}")
print("First available models:")
print(free_models[:10])

Number of available choices: 20
First available models:
['openrouter/free', 'cohere/north-mini-code:free', 'dots-studio/dots-3-note-preview:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'inclusionai/ling-3.0-flash-fin:free', 'inclusionai/ling-3.0-flash-sante:free', 'inclusionai/ling-3.0-flash-vl:free', 'liquid/lfm-2.5-2.6b:free', 'nex-agi/nex-n2.5-mini:free']


## How the OpenRouter request works

The chatbot sends:

- The selected model.
- The conversation history.
- The current user message.

The request is sent to OpenRouter's chat-completions endpoint.

The API key is read from the local environment and is not displayed or committed.

In [5]:
def ask_openrouter(messages, model):
    """
    Send messages to the selected OpenRouter model
    and return the model's response.
    """

    if not OPENROUTER_API_KEY:
        return "Error: OpenRouter API key was not found."

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://jekacode.africa",
        "X-Title": "Week 03 Educational AI Chatbot",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages
    }

    try:
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=60
        )

        response.raise_for_status()

        result = response.json()

        return result["choices"][0]["message"]["content"].strip()

    except requests.HTTPError as error:
        return f"API error: {error}"

    except requests.RequestException as error:
        return f"Network error: {error}"

    except KeyError:
        return "The API returned an unexpected response format."

    except Exception as error:
        return f"Unexpected error: {error}"

## Normal chatbot function

The normal chatbot:

1. Receives the user's message.
2. Converts previous messages into the format expected by the API.
3. Adds the new user message.
4. Sends the conversation to the selected model.
5. Displays the model's response.

In [6]:
def chat_fn(message, history, model):
    """
    Handle normal chatbot conversations.
    """

    messages = []

    for item in history or []:

        # Newer Gradio history format
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")

            if role in ["user", "assistant"] and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

        # Older Gradio history format
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item

            if user_message:
                messages.append({
                    "role": "user",
                    "content": str(user_message)
                })

            if assistant_message:
                messages.append({
                    "role": "assistant",
                    "content": str(assistant_message)
                })

    messages.append({
        "role": "user",
        "content": message
    })

    return ask_openrouter(messages, model)

## Cross-analysis feature

The cross-analysis feature sends the same question to multiple selected models.

It then asks one selected model to compare the responses.

The analysis includes:

- A summary of the responses.
- Similarities.
- Differences.
- Strengths.
- Weaknesses.
- The strongest answer.
- A recommendation.

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def get_model_response(model, message):
    return ask_openrouter(
        messages=[{"role": "user", "content": message}],
        model=model
    )


def cross_analyze(message, selected_models, progress=gr.Progress()):
    if not message or not message.strip():
        return "⚠️ Please enter a question."

    if not selected_models or len(selected_models) < 2:
        return "⚠️ Please select at least two models."

    # Use at most 4 models
    selected_models = selected_models[:4]

    progress(0, desc="Starting cross-analysis...")

    results = {}

    with ThreadPoolExecutor(max_workers=len(selected_models)) as executor:

        futures = {
            executor.submit(
                get_model_response,
                model,
                message
            ): model
            for model in selected_models
        }

        completed = 0

        for future in as_completed(futures):
            model = futures[future]

            try:
                results[model] = future.result()
            except Exception as error:
                results[model] = f"ERROR: {error}"

            completed += 1
            progress(
                completed / (len(selected_models) + 1),
                desc=f"Testing models: {completed}/{len(selected_models)}"
            )

    # Keep only successful responses
    successful = {}

    for model, response in results.items():
        if response and not response.startswith(("❌", "⚠️", "API error", "ERROR")):
            successful[model] = response

    if not successful:
        return "❌ None of the selected models returned a usable response."

    individual_responses = []

    for model, response in successful.items():
        individual_responses.append(
            f"### Model: `{model}`\n\n{response}"
        )

    responses_text = "\n\n---\n\n".join(individual_responses)

    # Use OpenRouter's free router as the evaluator
    analysis_prompt = f"""
Compare these AI responses.

QUESTION:
{message}

RESPONSES:
{responses_text}

Give:

## Overall Summary

## Similarities

## Differences

## Strengths

## Weaknesses

## Best Response

## Final Recommendation

Be concise and explain your reasoning.
"""

    progress(
        len(selected_models) / (len(selected_models) + 1),
        desc="Generating comparison..."
    )

    final_analysis = ask_openrouter(
        messages=[
            {
                "role": "user",
                "content": analysis_prompt
            }
        ],
        model="openrouter/free"
    )

    progress(1.0, desc="Cross-analysis complete.")

    return (
        "# Cross-Analysis Results\n\n"
        + responses_text
        + "\n\n---\n\n"
        + "## Comparative Analysis\n\n"
        + final_analysis
    )

## User interface

The interface contains two tabs:

### Normal Chat

The user selects one model and chats with it.

### Cross-Analysis

The user selects at least two models, enters a question, and compares their answers.

In [8]:
default_model = free_models[0] if free_models else "openrouter/free"

with gr.Blocks(title="Week 03 AI Chatbot") as demo:

    gr.Markdown(
        """
        # AI Chatbot — OpenRouter Free Models

        Chat with one model or compare multiple models.
        """
    )

    with gr.Tab("Normal Chat"):

        normal_model_picker = gr.Dropdown(
            choices=free_models,
            value=default_model,
            label="Select a model"
        )

        gr.ChatInterface(
            fn=chat_fn,
            additional_inputs=[normal_model_picker],
            title="Normal Chat",
            description="Select a model and start chatting."
        )

    with gr.Tab("Cross-Analysis"):

        analysis_model_picker = gr.Dropdown(
            choices=free_models,
            value=free_models[:3] if len(free_models) >= 3 else free_models,
            multiselect=True,
            label="Select at least two models"
        )

        analysis_question = gr.Textbox(
            label="Question",
            placeholder="Enter a question to compare across models..."
        )

        analyze_button = gr.Button("Run Cross-Analysis")

        analysis_output = gr.Markdown()

        analyze_button.click(
            fn=cross_analyze,
            inputs=[
                analysis_question,
                analysis_model_picker
            ],
            outputs=analysis_output
        )


if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
analysis_model_picker = gr.Dropdown(
    choices=free_models,
    value=[
        "openrouter/free",
        "nex-agi/nex-n2.5-mini:free"
    ],
    multiselect=True,
    label="Select 2–4 models"
)